# Gold — Matching Engine

Reads silver, applies the matching rules and writes:

- `gold.matched_pairs` - partner == internal rows, with the match_method and catch is_late_arrival pair.
- `gold.breaks_raw` - all rows that didn't match cleanly, tagged with
  `break_category`.

**Rule order**

1. `exact_match_on_txn_id` — inner join on (operator, partner_txn_id, txn_type)
2. `fallback_match_composite` — for rows with NULL partner_txn_id on the internal side
3. `flag_late_arrivals` — annotates matched set (orthogonal, not a category)
4. `classify_amount_mismatch` — among matched pairs, flag amount disagreements
5. `detect_orphan_churn` — among matched pairs, flag churned-but-billing users
6. `unmatched_partner_is_missing_on_platform` — what remains on partner side
7. `unmatched_internal_is_missing_at_partner` — what remains on internal side

Order matters and is defended in the case study README.

**Inputs**: `silver.partner_events`, `silver.internal_events`, `ods.user_churn_events`
**Outputs**: `gold.matched_pairs`, `gold.breaks_raw`


## Parameters

In [1]:
silver_partner_table = "silver_partner_events"
silver_internal_table = "silver_internal_events"
churn_table = "user_churn_events"
matched_table = "gold_matched_pairs"
breaks_table = "gold_breaks_raw"

# Tunable rule parameters
AMOUNT_TOLERANCE = 0.05                    
FALLBACK_TIME_WINDOW_SEC = 5 * 60
LATE_ARRIVAL_THRESHOLD_DAYS = 1 


StatementMeta(, 94b5a705-6383-41db-9ed7-c3bf45ce79e3, 3, Finished, Available, Finished, False)

## Imports

In [58]:
from pyspark.sql import DataFrame, functions as F, types as T
from delta.tables import DeltaTable
from dataclasses import dataclass, field

@dataclass
class RuleResult:
    rule_name: str
    matched_pairs: DataFrame
    partner_remaining: DataFrame
    internal_remaining: DataFrame
    breaks: DataFrame


StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 93, Finished, Available, Finished, False)

## Load Silver

In [2]:
partner_df = spark.table(silver_partner_table).cache()
internal_df = spark.table(silver_internal_table).cache()
churn_df = spark.table(churn_table)

StatementMeta(, 94b5a705-6383-41db-9ed7-c3bf45ce79e3, 4, Finished, Available, Finished, False)

## Rule 1 - exact match on partner_txn_id

both sides know the same `partner_txn_id` for the same operator and txn_type.

In [60]:
def rule_exact_match_on_txn_id(partner: DataFrame, internal: DataFrame) -> RuleResult:
    p = partner.filter(F.col("partner_txn_id").isNotNull()).alias("p")
    i = internal.filter(F.col("partner_txn_id").isNotNull()).alias("i")

    pairs = (p.join(i,
            (F.col("p.operator_code") == F.col("i.operator_code")) &
            (F.col("p.partner_txn_id") == F.col("i.partner_txn_id")) &
            (F.col("p.txn_type") == F.col("i.txn_type")),
            "inner")
        .select(
            F.col("p.operator_code").alias("operator_code"),
            F.col("p.partner_txn_id").alias("partner_txn_id"),
            F.col("p.txn_type").alias("txn_type"),
            F.col("p.partner_account_id").alias("partner_account_id"),
            F.col("p.amount").alias("amount_partner"),
            F.col("p.currency").alias("currency"),
            F.col("p.txn_ts_utc").alias("txn_ts_utc_partner"),
            F.col("p.business_date").alias("business_date_partner"),
            F.col("p.ingestion_date").alias("ingestion_date_partner"),
            F.col("p.source_file").alias("source_file"),
            F.col("i.internal_id").alias("internal_id"),
            F.col("i.internal_id_type").alias("internal_id_type"),
            F.col("i.user_id").alias("user_id"),
            F.col("i.status").alias("status"),
            F.col("i.amount").alias("amount_internal"),
            F.col("i.txn_ts_utc").alias("txn_ts_utc_internal"),
            F.col("i.business_date").alias("business_date_internal"),
            F.col("i.ingestion_date").alias("ingestion_date_internal"),
            F.col("i.source_table").alias("source_table"),
            F.lit("exact_txn_id").alias("match_method"),
        ))

    matched_keys = pairs.select("operator_code", "partner_txn_id", "txn_type").distinct()

    partner_remaining = (partner.alias("p").join(
            matched_keys.alias("k"),
            (F.col("p.operator_code") == F.col("k.operator_code")) &
            (F.col("p.partner_txn_id") == F.col("k.partner_txn_id")) &
            (F.col("p.txn_type") == F.col("k.txn_type")),
            "left_anti")
        .select("p.*"))

    internal_remaining = (internal.alias("i").join(
            matched_keys.alias("k"),
            (F.col("i.operator_code") == F.col("k.operator_code")) &
            (F.col("i.partner_txn_id") == F.col("k.partner_txn_id")) &
            (F.col("i.txn_type") == F.col("k.txn_type")),
            "left_anti")
        .select("i.*"))

    return RuleResult("exact_match_on_txn_id", pairs,
                      partner_remaining, internal_remaining,
                      spark.createDataFrame([], "string: string").limit(0))


StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 95, Finished, Available, Finished, False)

## Rule 2  - fallback match on composite key 

For internal rows where  `partner_txn_id` is missing

If a single internal row has 2+ candidate partner matches in the window, we refuse to pair (no match).


In [63]:
def rule_fallback_match_composite(partner: DataFrame, internal: DataFrame,
                                  user_account_map: DataFrame) -> RuleResult:
    
    i_missing = internal.filter(F.col("partner_txn_id").isNull())

    i_resolved = (i_missing.alias("i").join(
            user_account_map.alias("um"),
            F.col("i.user_id") == F.col("um.user_id"),
            "left")
        .select("i.*", F.col("um.partner_account_id").alias("expected_account")))

    p = partner
    i = i_resolved.alias("i")
    p = p.select([F.col(c).alias(f"{c}_p") for c in p.columns]).alias("p")

    window_sec = FALLBACK_TIME_WINDOW_SEC
    tolerance  = AMOUNT_TOLERANCE

    candidates = (p.join(i,
            (F.col("p.operator_code_p") == F.col("i.operator_code")) &
            (F.col("p.partner_account_id_p") == F.col("i.expected_account")) &
            (F.col("p.txn_type_p") == F.col("i.txn_type")) &
            (F.abs(F.unix_timestamp(F.col("p.txn_ts_utc_p")) -
                   F.unix_timestamp(F.col("i.txn_ts_utc"))) <= window_sec) &
            (F.col("i.amount").isNull() | F.col("p.amount_p").isNull() |
             (F.abs(F.col("p.amount_p") - F.col("i.amount")) <= tolerance)),
            "inner"))
            
    counts = (candidates
        .groupBy(F.col("i.operator_code").alias("op"),
                 F.col("i.internal_id").alias("iid"))
        .agg(F.count("*").alias("n_candidates")))

    unambiguous = (candidates.alias("c").join(counts.alias("ct"),
            (F.col("c.operator_code") == F.col("ct.op")) &
            (F.col("c.internal_id")   == F.col("ct.iid")),
            "inner")
        .filter(F.col("ct.n_candidates") == 1))

    pairs = unambiguous.select(
        F.col("c.operator_code_p").alias("operator_code"),
        F.col("c.partner_txn_id_p").alias("partner_txn_id"),
        F.col("c.txn_type_p").alias("txn_type"),
        F.col("c.partner_account_id_p").alias("partner_account_id"),
        F.col("c.amount_p").alias("amount_partner"),
        F.col("c.currency_p").alias("currency"),
        F.col("c.txn_ts_utc_p").alias("txn_ts_utc_partner"),
        F.col("c.business_date_p").alias("business_date_partner"),
        F.col("c.ingestion_date_p").alias("ingestion_date_partner"),
        F.col("c.source_file_p").alias("source_file"),
        F.col("c.internal_id").alias("internal_id"),
        F.col("c.internal_id_type").alias("internal_id_type"),
        F.col("c.user_id").alias("user_id"),
        F.col("c.status").alias("status"),
        F.col("c.amount").alias("amount_internal"),
        F.col("c.txn_ts_utc").alias("txn_ts_utc_internal"),
        F.col("c.business_date").alias("business_date_internal"),
        F.col("c.ingestion_date").alias("ingestion_date_internal"),
        F.col("c.source_table").alias("source_table"),
        F.lit("fallback_composite").alias("match_method"),
    )

    # Consumed rows: anti-join back against partner and internal.
    matched_partner_keys = pairs.select("operator_code", "partner_txn_id").distinct()
    matched_internal_keys = pairs.select(
        F.col("operator_code"), F.col("internal_id")).distinct()

    partner_remaining = (partner.alias("p").join(
            matched_partner_keys.alias("k"),
            (F.col("p.operator_code") == F.col("k.operator_code")) &
            (F.col("p.partner_txn_id") == F.col("k.partner_txn_id")),
            "left_anti")
        .select("p.*"))

    internal_remaining = (internal.alias("i").join(
            matched_internal_keys.alias("k"),
            (F.col("i.operator_code") == F.col("k.operator_code")) &
            (F.col("i.internal_id") == F.col("k.internal_id")),
            "left_anti")
        .select("i.*"))
    # return 0
    return RuleResult("fallback_match_composite", pairs,
                      partner_remaining, internal_remaining,
                      spark.createDataFrame([], "string: string").limit(0))


StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 98, Finished, Available, Finished, False)

## late arrivals

A pair where the partner row's ingestion_date is more than N days after its business_date is flagged.


In [64]:
def rule_flag_late_arrivals(matched: DataFrame) -> DataFrame:
    return matched.withColumn(
        "is_late_arrival",
        F.datediff(F.col("ingestion_date_partner"), F.col("business_date_partner"))
            > LATE_ARRIVAL_THRESHOLD_DAYS)


StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 99, Finished, Available, Finished, False)

## Rule 4 - amount mismatch

Among matched pairs, flag rows where amounts disagree beyond tolerance.


In [65]:
def rule_classify_amount_mismatch(matched: DataFrame) -> DataFrame:
    breaks = (matched
        .filter(F.col("amount_partner").isNotNull() &
                F.col("amount_internal").isNotNull() &
                (F.abs(F.col("amount_partner") - F.col("amount_internal")) > AMOUNT_TOLERANCE))
        .withColumn("break_category", F.lit("amount_mismatch"))
        .withColumn("variance", F.col("amount_partner") - F.col("amount_internal")))
    return breaks


StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 100, Finished, Available, Finished, False)

## Rule 5 - detect orphan churn

find any where the user has a churn event that predates the transaction. Partner is still billing a user the platform
has marked churned.


In [66]:
def rule_detect_orphan_churn(matched: DataFrame, churn: DataFrame) -> DataFrame:
    breaks = (matched.alias("m").join(
            F.broadcast(churn.alias("c")),
            (F.col("m.operator_code") == F.col("c.operator_code")) &
            (F.col("m.user_id")       == F.col("c.user_id")),
            "inner")
        .filter(F.col("c.churn_ts_utc") < F.col("m.txn_ts_utc_partner"))
        .select(
            "m.*",
            F.col("c.churn_ts_utc"),
            F.col("c.churn_reason"))
        .withColumn("break_category", F.lit("orphan_churn")))
    return breaks


StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 101, Finished, Available, Finished, False)

## Rules 6 & 7 - missing on each side

Whatever's left in `partner_remaining` after both matching rules =
the partner billed but the platform has no record (missing_on_platform).
Whatever's left in `internal_remaining` = the platform granted
entitlement without a corresponding partner charge (missing_at_partner).


In [67]:
def rule_unmatched_partner_is_missing_on_platform(partner_remaining: DataFrame) -> DataFrame:
    return partner_remaining.withColumn("break_category", F.lit("missing_on_platform"))

def rule_unmatched_internal_is_missing_at_partner(internal_remaining: DataFrame) -> DataFrame:
    return internal_remaining.withColumn("break_category", F.lit("missing_at_partner"))


StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 102, Finished, Available, Finished, False)

In [68]:
# sample dim user for partner id
USER_ACCOUNT_DATA = [
    ("U-1",   "923001111111"), ("U-2",   "923002222222"), ("U-3", "923003333333"),
    ("U-5",   "923005555555"), ("U-6",   "923006666666"),
    ("U-7",   "923007777777"), ("U-8",   "923008888888"), ("U-9", "923009999999"),
    ("U-999", "923099999999"),
    ("U-101", "254700111111"), ("U-102", "254700222222"), ("U-103", "254700333333"),
]
user_account_map = spark.createDataFrame(
    USER_ACCOUNT_DATA, schema="user_id STRING, partner_account_id STRING")


StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 103, Finished, Available, Finished, False)

+-------+------------------+
|user_id|partner_account_id|
+-------+------------------+
|    U-1|      923001111111|
|    U-2|      923002222222|
|    U-3|      923003333333|
|    U-5|      923005555555|
|    U-6|      923006666666|
+-------+------------------+
only showing top 5 rows



## Run the fold

Apply rules in priority order. Each rule consumes off the front and
hands the remainder to the next.


In [69]:
r1 = rule_exact_match_on_txn_id(partner_df, internal_df)
r2 = rule_fallback_match_composite(r1.partner_remaining, r1.internal_remaining,
                                    user_account_map)

matched = r1.matched_pairs.unionByName(r2.matched_pairs)
matched = rule_flag_late_arrivals(matched).cache()

# classifying breaks
amount_breaks = rule_classify_amount_mismatch(matched)
orphan_breaks = rule_detect_orphan_churn(matched, churn_df)
missing_part = rule_unmatched_partner_is_missing_on_platform(r2.partner_remaining)
missing_int = rule_unmatched_internal_is_missing_at_partner(r2.internal_remaining)

StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 105, Finished, Available, Finished, False)

Counts after the fold:
  matched (exact):       10
  matched (fallback):    1
  matched (total):       11
  amount_mismatch:       2
  orphan_churn:          1
  missing_on_platform:   1
  missing_at_partner:    1


## Structure breaks

Different break categories carry different columns. We unify them by
projecting each into a common structure.


In [70]:
def envelope(df: DataFrame, source: str) -> DataFrame:

    is_partner_side  = "amount_partner"  in df.columns
    is_internal_side = "amount_internal" in df.columns

    if is_partner_side and is_internal_side:
        partner_amt_expr  = F.col("amount_partner")
        internal_amt_expr = F.col("amount_internal")
    elif is_partner_side:
        partner_amt_expr  = F.col("amount_partner")
        internal_amt_expr = F.lit(None).cast("double")
    elif "amount" in df.columns and source == "missing_on_platform":
        partner_amt_expr  = F.col("amount")
        internal_amt_expr = F.lit(None).cast("double")
    elif "amount" in df.columns and source == "missing_at_partner":
        partner_amt_expr  = F.lit(None).cast("double")
        internal_amt_expr = F.col("amount")
    else:
        partner_amt_expr  = F.lit(None).cast("double")
        internal_amt_expr = F.lit(None).cast("double")

    cols = {
        "operator_code":   F.col("operator_code"),
        "partner_txn_id":  F.col("partner_txn_id") if "partner_txn_id" in df.columns else F.lit(None).cast("string"),
        "internal_id":     F.col("internal_id") if "internal_id" in df.columns else F.lit(None).cast("string"),
        "break_category":  F.col("break_category"),
        "business_date":   (F.col("business_date_partner") if "business_date_partner" in df.columns
                            else F.col("business_date")),
        "ingestion_date":  (F.col("ingestion_date_partner") if "ingestion_date_partner" in df.columns
                            else F.col("ingestion_date")),
        "partner_amount":  partner_amt_expr,
        "internal_amount": internal_amt_expr,
        "variance":        F.col("variance") if "variance" in df.columns else F.lit(None).cast("double"),
        "source_rule":     F.lit(source),
    }
    return df.select(*[v.alias(k) for k, v in cols.items()])

breaks_raw = (envelope(amount_breaks,  "amount_mismatch")
    .unionByName(envelope(orphan_breaks, "orphan_churn"))
    .unionByName(envelope(missing_part,  "missing_on_platform"))
    .unionByName(envelope(missing_int,   "missing_at_partner")))

print(f"breaks_raw total rows: {breaks_raw.count()}")
breaks_raw.groupBy("break_category").count().show()


StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 106, Finished, Available, Finished, False)

breaks_raw total rows: 5
+-------------------+-----+
|     break_category|count|
+-------------------+-----+
|    amount_mismatch|    2|
|       orphan_churn|    1|
|missing_on_platform|    1|
| missing_at_partner|    1|
+-------------------+-----+



## Write output


In [71]:
def write_delta(df: DataFrame, table_name: str, partition_cols: list):
    (df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy(*partition_cols)
        .format("delta")
        .saveAsTable(table_name))
    print(f"Wrote {table_name} ({df.count()} rows)")

write_delta(matched,    matched_table, ["business_date_partner", "operator_code"])
write_delta(breaks_raw, breaks_table,  ["business_date",         "operator_code"])


StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 107, Finished, Available, Finished, False)

Wrote gold_matched_pairs (11 rows)
Wrote gold_breaks_raw (5 rows)


## Sanity checks

In [77]:
%%sql

select * from gold_breaks_raw

StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 113, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 10 fields>

In [74]:
%%sql

select * from gold_matched_pairs

StatementMeta(, 362686a5-f254-4592-a98c-8ecdd0b13091, 110, Finished, Available, Finished, False)

<Spark SQL result set with 11 rows and 21 fields>